# 02: Source selection

**Phase 2: which dataset do I build from?**

The choice is between raw `ts` (circuit-level telemetry) and `structured_data`
(currently `structured_data_v2_flex_included`, site-level, PV-only by
construction). Phase 1 already turned up every other candidate table in the
catalogue and ruled all of them out on grain alone (derived model/conformance
outputs, not signal) — that finding is not re-litigated here, only cited.

The comparison logic lives in `lib/ami_sources.py`, not in this notebook —
`SourceCandidate`, `verify_is_pv_only`, `compare_circuit_and_row_shares`,
`build_comparison_table`, `recommend`, all pure and unit-tested against
synthetic frames. This notebook gathers the evidence those functions need and
narrates the result.

---

### Expected Athena scan

| Cell | What it does | Expected scan |
|---|---|---|
| Fresh partition probe (`ts` + 3 `structured_data` variants) | `$partitions` metadata, same free mechanism as notebook 01 | ~4 x 10 MB minimum, negligible |
| `information_schema.columns` for `solar_analytics_iceberg` | one metadata call, reused for two schema checks | ~10 MB minimum |
| `meta_up23c` circuit counts by `is_pv` | full scan of a small (424k-row) dimension table | well under 10 MB, rounds up to the minimum |
| 5 sample circuit rows from `meta_up23c` | same small table | 10 MB minimum |
| **Real load-circuit sample from `ts`** (flagged below) | one month, `is_pv=false`, filtered to 5 specific `circuit_id`s | **usually small — Iceberg's per-file `circuit_id` min/max stats (visible in notebook 01's raw `$partitions` output) let Athena skip most files. Worst case if that pruning doesn't help: ~8.5 GB (one month's full `is_pv=false` partition, all 17 postcode buckets). At Sydney's ~AUD $8/TB that worst case is still only ~7 cents — flagged because you asked to know the BYTES regardless of dollar cost, not because it's expensive.** Set `RUN_SAMPLE_QUERY = False` in that cell to skip it.

Total expected: well under 1 GB in the routine part; the one flagged cell could
reach single-digit GB in its unlikely worst case. Nothing here is anywhere near
the "tell me before scanning more than a few GB" threshold in dollar terms, but
the byte volume is flagged as requested regardless.


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as Config
from bms_sa_review.ami_data_analysis.lib import ami_athena as Athena
from bms_sa_review.ami_data_analysis.lib import ami_inventory as Inventory
from bms_sa_review.ami_data_analysis.lib import ami_sources as Sources

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

Athena.reset_scan_log()
Athena.require_credentials()
print("Credentials OK. Starting source selection.")


Credentials OK. Starting source selection.


## 1. The choice, restated

- **`ts`** (`solar_analytics_iceberg.ts`) — raw circuit-level telemetry, joined
  to `meta_up23c` for site_id and circuit_polarity. Confirmed in Phase 1:
  16,345,254,058 rows, ~457 GB compressed, 24 months (2024-01 .. 2025-12), and
  **52.1% of rows are `is_pv=false` (load)** — the larger half, not a discarded
  remainder.
- **`structured_data_v2_flex_included`** (`Config.TABLES["structured_data"]`) —
  site-level, built by `build_structured_data.py`, which filters
  `ts.is_pv = True` before ever summing to site level.
- **Everything else Phase 1 found** (`all_uncurtailedpv*`, `conformance_*`,
  `pv_ghi_norm_model*`, `split_days*`, `lso_*`) — derived analytical outputs of
  the Stage 1/2 pipeline, not raw signal at any grain. Not re-examined here;
  see notebook 01 section 10 for why.

The brief calls the `is_pv` question "the single most consequential fact in
this phase." Phase 1 already answered it from partition metadata and from
reading `build_structured_data.py`. This notebook reconfirms it live,
independently, and does not simply cite the earlier chat output.


## 2. Is `structured_data` structurally capable of carrying load, at all?

Independent of any row it contains: does its SCHEMA even have an `is_pv`
column? If not, no query against it could ever recover load data — the
exclusion happened upstream, when the table was built.

One `information_schema.columns` call covers the whole `solar_analytics_iceberg`
database; reused below for both `ts` and the structured_data target rather than
querying twice.


In [2]:
sai_schema = Inventory.column_inventory(Config.SAI)

structured_target = Config.TABLES["structured_data"]

ts_check = Sources.verify_is_pv_only(sai_schema[sai_schema.table_name == "ts"])
structured_check = Sources.verify_is_pv_only(
    sai_schema[sai_schema.table_name == structured_target]
)

print(f"ts:                 {ts_check}")
print(f"{structured_target}: {structured_check}")

assert ts_check["is_pv_only"] is False, (
    "ts has no is_pv column?! That contradicts notebook 01's partition finding -- "
    "stop and investigate before trusting anything below."
)


ts:                 {'is_pv_only': False, 'reason': '`is_pv` column present -- both signals may coexist here'}
structured_data_v2_flex_included: {'is_pv_only': True, 'reason': 'no `is_pv` column in the schema -- the table cannot carry both signals'}


**Read this before continuing.** If `structured_check["is_pv_only"]` above
came back anything other than `True`, the schema has changed since Phase 1 and
the rest of this notebook's conclusion should not be trusted as written — tell
me rather than proceeding.


## 3. Fresh, self-contained size and row counts

This notebook does not assume notebook 01's kernel state survived. Same free
`$partitions` mechanism, scoped to just the four tables this phase compares, and
cross-checked against Phase 1's recorded findings — a mismatch would mean the
catalogue changed since Phase 1 ran.


In [3]:
catalog = Inventory.glue_inventory()

fresh_totals, fresh_raw, fresh_log = Inventory.probe_partitions(
    catalog,
    only=["ts", "structured_data", "structured_data_v2", "structured_data_v2_flex_included"],
)
display(fresh_log[["table", "n_partitions", "declared_partition_keys",
                    "actual_partition_columns", "error"]])


  solar_analytics_iceberg.structured_data               24 partitions,   1,022,900,647 rows  [partitioning hidden from Glue]
  solar_analytics_iceberg.structured_data_v2            24 partitions,     841,491,009 rows  [partitioning hidden from Glue]
  solar_analytics_iceberg.structured_data_v2_flex_included    24 partitions,     871,655,350 rows  [partitioning hidden from Glue]
  solar_analytics_iceberg.ts                           816 partitions,  16,345,254,058 rows  [partitioning hidden from Glue]

Note: 4 Iceberg table(s) are partitioned but Glue declares no PartitionKeys for them -- a known Iceberg-on-Glue quirk, not a data problem. Real partition columns, from $partitions:
  - solar_analytics_iceberg.structured_data: year, month
  - solar_analytics_iceberg.structured_data_v2: year, month
  - solar_analytics_iceberg.structured_data_v2_flex_included: year, month
  - solar_analytics_iceberg.ts: year, month, is_pv


,table,n_partitions,declared_partition_keys,actual_partition_columns,error
0,solar_analytics_iceberg.structured_data,24,(none declared in Glue),"year, month",
1,solar_analytics_iceberg.structured_data_v2,24,(none declared in Glue),"year, month",
2,solar_analytics_iceberg.structured_data_v2_fle...,24,(none declared in Glue),"year, month",
3,solar_analytics_iceberg.ts,816,(none declared in Glue),"year, month, is_pv",


In [4]:
ts_key = f"{Config.SAI}.ts"
structured_key = f"{Config.SAI}.{structured_target}"

ts_stats = fresh_totals.get(ts_key, {})
if ts_stats.get("n_rows") == Config.TS_TOTAL_ROWS:
    print(f"ts row count matches Phase 1's recorded finding: {Config.TS_TOTAL_ROWS:,}")
else:
    print(f"MISMATCH: ami_config recorded {Config.TS_TOTAL_ROWS:,} rows for ts, "
          f"a fresh probe now shows {ts_stats.get('n_rows')!r}. The catalogue has "
          "changed since Phase 1 -- update ami_config.py before trusting anything "
          "downstream of this.")

structured_stats = fresh_totals.get(structured_key, {})
print(f"\n{structured_target}: {structured_stats.get('n_rows', 0):,} rows, "
      f"{Athena.fmt_bytes(structured_stats.get('size_bytes'))}")


ts row count matches Phase 1's recorded finding: 16,345,254,058

structured_data_v2_flex_included: 871,655,350.0 rows, 36.62 GB


## 4. Circuit-count share vs row-volume share

Phase 1's 52.1%/47.9% split is by ROW volume across 24 months. Here we get the
split by CIRCUIT COUNT from `meta_up23c` — a different measure of the same
fleet, and it need not agree exactly (a circuit reporting less completely
contributes fewer rows than its circuit-count share implies). `meta_up23c` is a
small dimension table — no partition predicate needed, cheap regardless.


In [5]:
circuit_counts = Athena.aq(
    """
    SELECT is_pv, count(*) AS n_circuits, count(DISTINCT site_id) AS n_sites
    FROM meta_up23c
    GROUP BY is_pv
    """,
    database=Config.SAI, label="meta_up23c circuit counts by is_pv",
)
display(circuit_counts)

share_comparison = Sources.compare_circuit_and_row_shares(
    circuit_counts,
    ts_rows_false=Config.TS_ROWS_BY_IS_PV["is_pv=false (load)"],
    ts_rows_true=Config.TS_ROWS_BY_IS_PV["is_pv=true (pv)"],
)
display(share_comparison)


,is_pv,n_circuits,n_sites
0,True,27108,16148
1,False,33462,15167


,label,n_circuits,share_of_circuits,n_sites,n_rows_in_ts,share_of_ts_rows
0,is_pv=true (pv),27108,0.447548,16148,7823538598,0.478643
1,is_pv=false (load),33462,0.552452,15167,8521715460,0.521357


## 5. A real sample of load-circuit data

Statistics establish that load rows exist; they don't show what they look
like. This pulls a handful of real `is_pv=false` circuits and a month of their
actual `power`/`voltage`/`energy_reactive` values.

**Cost note — read before running the next cell.** The dimension-table lookup
below is free-equivalent. The `ts` sample after it adds an explicit
`circuit_id IN (...)` filter — Iceberg tracks per-file min/max `circuit_id`
statistics (visible directly in notebook 01's raw `$partitions` output, in the
`data` column), so Athena can usually skip most files entirely rather than
scanning the whole month. Usually cheap; not formally guaranteed. Worst case
(no pruning at all): ~8.5 GB, ~7 cents AUD. Set `RUN_SAMPLE_QUERY = False` below
to skip this cell entirely if you'd rather not risk it.


In [6]:
sample_circuits = Athena.aq(
    """
    SELECT circuit_id, site_id, circuit_polarity, circuit_type, ac_capacity_kw, s_99
    FROM meta_up23c
    WHERE is_pv = false
    LIMIT 5
    """,
    database=Config.SAI, label="sample is_pv=false circuits",
)
display(sample_circuits)


,circuit_id,site_id,circuit_polarity,circuit_type,ac_capacity_kw,s_99
0,254727,2083958230,1,load_air_conditioner,5.0,4.930158
1,276769,248529252,1,ac_load_net,5.0,3.980736
2,276768,248529252,1,ac_load_net,5.0,3.980736
3,276765,248529252,1,load_pool,5.0,3.980736
4,298419,804338564,1,load_hot_water,5.0,4.819746


In [7]:
RUN_SAMPLE_QUERY = True  # set False to skip -- see the cost note above

if RUN_SAMPLE_QUERY:
    circuit_ids = ", ".join(str(int(c)) for c in sample_circuits.circuit_id)
    ts_sample = Athena.aq(
        f"""
        SELECT circuit_id, t_stamp, power, voltage, energy_reactive
        FROM ts
        WHERE year = 2025 AND month = 6 AND is_pv = false
          AND circuit_id IN ({circuit_ids})
        ORDER BY circuit_id, t_stamp
        LIMIT 50
        """,
        database=Config.SAI, label="ts sample, is_pv=false circuits",
    )
    merged = ts_sample.merge(
        sample_circuits[["circuit_id", "circuit_polarity", "circuit_type"]],
        on="circuit_id", how="left",
    )
    # Illustrative only -- Phase 3 is where the sign convention is decided and
    # written down once. Shown here so you can eyeball plausibility, not as a
    # methodological choice.
    merged["power_corrected_kw_illustrative"] = merged.power * merged.circuit_polarity / 1000
    display(merged)
else:
    print("RUN_SAMPLE_QUERY is False -- skipped.")


,circuit_id,t_stamp,power,voltage,energy_reactive,circuit_polarity,circuit_type,power_corrected_kw_illustrative
0,254727,2025-06-01 00:00:00,11.9867,242.95,-10.9547,1,load_air_conditioner,0.011987
1,254727,2025-06-01 00:05:00,11.9700,242.35,-10.9022,1,load_air_conditioner,0.01197
2,254727,2025-06-01 00:10:00,11.8600,242.70,-10.9397,1,load_air_conditioner,0.01186
3,254727,2025-06-01 00:15:00,11.8733,243.80,-11.0364,1,load_air_conditioner,0.011873
4,254727,2025-06-01 00:20:00,11.7267,241.80,-10.7983,1,load_air_conditioner,0.011727
5,254727,2025-06-01 00:25:00,11.7100,242.00,-10.7922,1,load_air_conditioner,0.01171
6,254727,2025-06-01 00:30:00,11.6933,241.80,-10.8378,1,load_air_conditioner,0.011693
7,254727,2025-06-01 00:35:00,11.6767,241.60,-10.8175,1,load_air_conditioner,0.011677
8,254727,2025-06-01 00:40:00,11.6567,242.25,-10.8750,1,load_air_conditioner,0.011657
9,254727,2025-06-01 00:45:00,11.6300,242.75,-10.9503,1,load_air_conditioner,0.01163


## 6. The trade-off table

Granularity, both signals, and cost — assembled from the evidence gathered
above via `Sources.build_comparison_table`, not hand-typed.


In [8]:
candidates = [
    Sources.SourceCandidate(
        name="raw `ts` + `meta_up23c`",
        grain="circuit, 5-min",
        has_load_signal=not ts_check["is_pv_only"],
        has_pv_signal=True,
        decomposable=True,  # circuit-level -- signals CAN be told apart, given a
                            # correct circuit-to-signal map (Phase 3's job, not yet done)
        n_rows=ts_stats.get("n_rows"),
        size_bytes=ts_stats.get("size_bytes"),
        cleanliness_notes=(
            "raw power (instantaneous W) and energy_reactive (5-min kvarh) resample "
            "differently -- see ami_config.SOURCE_COLUMN_UNITS; circuit_polarity sign "
            "correction required; meta_up23c fans out 2.47x over circuits, GROUP BY "
            "circuit_id + max(...) required before joining (Phase 1 finding)"
        ),
        complexity_notes=(
            "circuit-to-signal mapping and aggregate-circuit (double-counting) "
            "detection are NEW work -- the existing pipeline never needed this, "
            "because it only ever consumed the PV half"
        ),
    ),
    Sources.SourceCandidate(
        name=f"`{structured_target}` (site-level, PV-only)",
        grain="site, 5-min",
        has_load_signal=not structured_check["is_pv_only"],
        has_pv_signal=True,
        decomposable=False,  # even if load were present, one P_kw_norm column per
                             # site cannot be split back into components
        n_rows=structured_stats.get("n_rows"),
        size_bytes=structured_stats.get("size_bytes"),
        cleanliness_notes=(
            "already site-summed, capacity-normalized (p_kw_norm, not kW -- needs "
            "de-normalizing), clear-sky/GHI enriched, voltage-bounded -- far less "
            "post-processing needed IF it had what this project needs"
        ),
        complexity_notes="ruled out on has_load_signal / decomposable -- cost is moot",
    ),
]

comparison = Sources.build_comparison_table(candidates)
display(comparison)

verdict = Sources.recommend(candidates)
print("\nQualifying candidate(s):", verdict["qualifying"])
print("Excluded:")
for name, reason in verdict["excluded"].items():
    print(f"  - {name}: {reason}")


,candidate,grain,has_load_signal,has_pv_signal,decomposable,n_rows,size,full_scan_cost_aud,cleanliness,complexity
0,raw `ts` + `meta_up23c`,"circuit, 5-min",True,True,True,1.634525e+10,456.93 GB,3.57,raw power (instantaneous W) and energy_reactiv...,circuit-to-signal mapping and aggregate-circui...
1,`structured_data_v2_flex_included` (site-level...,"site, 5-min",False,True,False,8.716554e+08,36.62 GB,0.29,"already site-summed, capacity-normalized (p_kw...",ruled out on has_load_signal / decomposable --...



Qualifying candidate(s): ['raw `ts` + `meta_up23c`']
Excluded:
  - `structured_data_v2_flex_included` (site-level, PV-only): no load signal, not decomposable at this grain


## 7. What this cost


In [ ]:
display(Athena.scan_report())

8 queries, 74.61 MB scanned, ~AUD 0.0011 (billed at a 10.00 MB minimum per query)


,label,database,n_rows,seconds,scanned,scanned_bytes,cost,source
0,information_schema.columns [solar_analytics_ic...,solar_analytics_iceberg,445,9.96,40.82 KB,41804.0,0.0001,query_metadata
1,structured_data$partitions,solar_analytics_iceberg,24,3.20,13.92 KB,14256.0,0.0001,query_metadata
2,structured_data_v2$partitions,solar_analytics_iceberg,24,5.02,19.10 KB,19560.0,0.0001,query_metadata
3,structured_data_v2_flex_included$partitions,solar_analytics_iceberg,24,3.44,19.10 KB,19560.0,0.0001,query_metadata
4,ts$partitions,solar_analytics_iceberg,816,4.20,409.59 KB,419424.0,0.0001,query_metadata
5,meta_up23c circuit counts by is_pv,solar_analytics_iceberg,2,3.32,1.01 MB,1056199.0,0.0001,query_metadata
6,sample is_pv=false circuits,solar_analytics_iceberg,5,3.12,592.12 KB,606329.0,0.0001,query_metadata
7,"ts sample, is_pv=false circuits",solar_analytics_iceberg,50,2.92,72.53 MB,76052019.0,0.0006,query_metadata


## 8. Recommendation

**Build from raw `ts`, joined to `meta_up23c`. Do not build from
`structured_data` in any variant.**

This is not a close call decided on cost or cleanliness — it's a hard
disqualification. `structured_data` fails the one non-negotiable requirement
(a source must carry both `pv_generation` and `gross_load` at a grain fine
enough to separate them) on two independent grounds that would each be
sufficient alone:

1. **No `is_pv` column in its schema, confirmed live above.** Whatever
   `ts.is_pv = True` excluded when `build_structured_data.py` built this table,
   that exclusion is permanent and unrecoverable from the table as it exists.
2. **Even if load rows were somehow present, they'd be unusable.** The table
   stores one `p_kw_norm` column per site per interval — a single normalized
   scalar, not a per-circuit or per-type breakdown. There is no column
   structure that could ever separate load from PV, independent of the
   `is_pv` filter.

`ts` pays real costs for being the only qualifying source — costs this
notebook did not have to invent, because they were already visible in the
evidence gathered above:

- **Size**: ~457 GB compressed vs. `structured_data`'s much smaller footprint
  (section 3 has the live number). In Athena dollars this is nearly irrelevant
  — a full scan of all of `ts` is roughly the price shown in this notebook's
  comparison table, typically a few AUD — but it is the number Phase 4 sizes
  its local extract against.
- **Correctness risk, not cost, is the real price.** `structured_data` arrives
  clean: normalized, GHI-enriched, voltage-bounded. `ts` arrives raw: two
  columns that resample by different rules (`ami_config.SOURCE_COLUMN_UNITS`),
  a polarity sign to apply per circuit, and a metadata table that fans out
  2.47x over circuits and must be pre-aggregated before joining. None of this
  is new information — it was true the moment Phase 1 read `meta_up23c`'s row
  count — but it is the cost this recommendation is actually asking you to
  accept, and Phase 3 is where each piece gets resolved and pinned down in one
  place, per the module sketch.

Every other table Phase 1 found is excluded on grain alone (derived, not raw
signal) and isn't re-litigated here.

**This is what I recommend — confirm it, or overrule it, before Phase 3
starts.** If you confirm, I'll update `ami_config.SOURCE_CHOICE` from
`"UNRESOLVED"` to `"ts"` and mark it resolved, the same way Phase 1's findings
were only written into `ami_config` once you'd seen and agreed with them.
